**Notebook con el proyecto final del grupo Condores

Lectura de la información
Como las fuentes de datos son bastante pesadas 
    Licitaciones 42.3MB
    Ordenes de compra 520.8MB
se toma la decisión de subir las fuentes a google Drive y desde allí gestionar los permisos de descarga de la información

In [2]:
import os
import requests
import pandas as pd

# Tu nuevo ID de archivo
FILE_ID = '11fA5-HKOX0YmWflZdRFmuK7wKrzX1f05'
URL = f'https://drive.google.com/uc?export=download&id={FILE_ID}'
NOMBRE_LOCAL = 'licitaciones_final.parquet'

print("--- DESCARGANDO DESDE TU DRIVE ---")

try:
    # Descarga directa
    response = requests.get(URL, stream=True)
    with open(NOMBRE_LOCAL, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk: f.write(chunk)
            
    # Verificación de integridad
    peso_mb = os.path.getsize(NOMBRE_LOCAL) / (1024 * 1024)
    print(f"Archivo descargado: {peso_mb:.2f} MB")
    
    if peso_mb > 1.0:
        df_licitaciones = pd.read_parquet(NOMBRE_LOCAL, engine='pyarrow')
        print(f"✅ ¡ÉXITO! Licitaciones cargadas: {df_licitaciones.shape} filas.")
        display(df_licitaciones.head())
    else:
        print("❌ El archivo sigue siendo demasiado pequeño. Revisa que el permiso sea 'Cualquier persona con el enlace'.")

except Exception as e:
    print(f"❌ Error: {e}")


--- DESCARGANDO DESDE TU DRIVE ---
Archivo descargado: 42.33 MB
✅ ¡ÉXITO! Licitaciones cargadas: (826238, 4) filas.


,CodigoExterno,Nombre,CodigoEstado,FechaCierre
0,564162-187-L119,(id.48430) Particulas para Depto de Física,7,2020-01-01T17:04:01.527
1,1058134-381-L119,Medicamentos Enero 2020,7,2020-01-01T15:04:01.643
2,1663-111-L119,SERVICIO DE LABORATORIO DENTAL PARA PRÓTESIS M...,7,2020-01-01T21:44:01.157
3,2713-290-L119,CLASES DIRIGIDAS,7,2020-01-01T19:29:01.22
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00


In [5]:
import os
import requests
import pandas as pd

def descargar_archivo_drive(file_id, destino):
    # FORZAR LIMPIEZA: Si el archivo existe pero es pequeño (<1MB), se borra para re-descargar
    if os.path.exists(destino) and os.path.getsize(destino) < 1000000:
        os.remove(destino)
        print(f"🗑️ Eliminando archivo corrupto previo: {destino}")

    if os.path.exists(destino):
        print(f"✅ {destino} ya existe y parece válido.")
        return True
    
    print(f"Descargando {destino} (esta vez de forma robusta)...")
    URL = "https://docs.google.com"
    session = requests.Session()
    
    # 1. Primera petición para token
    response = session.get(URL, params={'id': file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break
            
    # 2. Petición final con confirmación
    if token:
        response = session.get(URL, params={'id': file_id, 'confirm': token}, stream=True)
    else:
        # Re-intento directo si no hay token (común en archivos <100MB)
        response = session.get(URL, params={'id': file_id}, stream=True)

    with open(destino, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk: f.write(chunk)
            
    return True

ids_ordenes = {
    '2020': '1ik7963B6cXGjIuYiRTrwP_ywXI7rOtVf',
    '2021': '1EkM03BjnbL3l0cLJW7nXRf6wL0j0YMCP',
    '2022': '1ibPVPltJ19vELEd_cMLh6Kf8uLtsxv5s',
    '2023': '1YxvMZZQcrIVPIeZxUwMh_GNXyM20ADHU',
    '2024': '1bWRQqVRxKp8Ly8RuLs14xhg3IxqZd4gs',
    '2025': '1UQXDjlVZ173eTk8EAYiwnCrk7DDrDQFu'
}

dataframes = []

for anio, f_id in ids_ordenes.items():
    nombre = f'ordenes_{anio}.parquet'
    if descargar_archivo_drive(f_id, nombre):
        try:
            # Forzamos motor pyarrow para evitar ambigüedades
            df_temp = pd.read_parquet(nombre, engine='pyarrow')
            dataframes.append(df_temp)
            print(f"   📊 Año {anio} CARGADO: {len(df_temp):,} filas.")
        except Exception as e:
            print(f"   ❌ Error al leer {nombre}: {e}")

if dataframes:
    df_final = pd.concat(dataframes, ignore_index=True)
    print(f"\n🚀 CONSOLIDACIÓN EXITOSA: {df_final.shape[0]:,} registros totales.")
    display(df_final.head())


🗑️ Eliminando archivo corrupto previo: ordenes_2020.parquet
Descargando ordenes_2020.parquet (esta vez de forma robusta)...
   ❌ Error al leer ordenes_2020.parquet: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
🗑️ Eliminando archivo corrupto previo: ordenes_2021.parquet
Descargando ordenes_2021.parquet (esta vez de forma robusta)...
   ❌ Error al leer ordenes_2021.parquet: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
🗑️ Eliminando archivo corrupto previo: ordenes_2022.parquet
Descargando ordenes_2022.parquet (esta vez de forma robusta)...
   ❌ Error al leer ordenes_2022.parquet: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
🗑️ Eliminando archivo corrupto previo: ordenes_2023.parque